# Seebeck coefficient counts by material family, across DFT-only databases

This notebook pulls entries from **DFT-only** (BoltzTraP/DFT-computed, not experimental) thermoelectric transport databases, classifies each entry into one of three families:

- **half_heusler** — ABC, 1:1:1, space group *F-43m* (216)
- **skutterudite** — M X₃ binary (CoSb₃-type) or filled R M₄X₁₂, space group *Im-3* (204)
- **bi2te3_family** — tetradymite A₂B₃ (Bi/Sb + Te/Se/S), space group *R-3m* (166)

and writes out **one CSV per (dataset, family) combination**, plus a combined summary table.

### Datasets covered
| Section | Dataset | Size | Needs API key? |
|---|---|---|---|
| A | Ricci et al. / Materials Project BoltzTraP (`boltztrap_mp`, via matminer) | 8,924 | No |
| B | JARVIS-DFT (`dft_3d`, via jarvis-tools) | ~76,000 (Seebeck subset ~22,190) | No |
| C (optional) | Full Ricci tabular (`ricci_boltztrap_mp_tabular`, 47,737 entries) + live Materials Project symmetry lookup | 47,737 | **Yes** — free key from https://materialsproject.org/api |

> ⚠️ **Important:** the family classification here is a **heuristic** based on space group + stoichiometry + element sets. It's a strong first pass, but always spot-check a sample of each output CSV (formula + space group columns are included) before treating counts as final. Some real half-Heuslers/skutterudites relax to slightly distorted structures with a different detected space group, so heuristic counts are a lower bound.

Run cells top to bottom. Everything is written to `/content/te_family_csvs/`.

In [1]:
# --- Install dependencies (Colab) ---
!pip install -q matminer pymatgen jarvis-tools mp-api pandas tqdm


[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# --- Imports & shared config ---
import os
import re
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

from pymatgen.core.composition import Composition
from pymatgen.symmetry.analyzer import SpacegroupAnalyzer

OUT_DIR = "/content/te_family_csvs"
os.makedirs(OUT_DIR, exist_ok=True)

FAMILIES = ["half_heusler", "skutterudite", "bi2te3_family"]

PNICTOGENS = {"Bi", "Sb", "As"}
CHALCOGENS = {"Te", "Se", "S"}


def classify_family(structure, symprec=0.1):
    """Heuristic classifier: returns one of FAMILIES or None.
    `structure` must be a pymatgen Structure.
    """
    try:
        comp = structure.composition
        reduced = comp.reduced_composition.get_el_amt_dict()
        n_el = len(reduced)
        vals = sorted(reduced.values())
        elements = set(reduced.keys())

        try:
            sga = SpacegroupAnalyzer(structure, symprec=symprec)
            sgnum = sga.get_space_group_number()
        except Exception:
            sgnum = None

        # ---- half-Heusler: ABC, 1:1:1, F-43m (216) ----
        if n_el == 3 and sgnum == 216:
            if abs(vals[0] - vals[1]) < 1e-2 and abs(vals[1] - vals[2]) < 1e-2:
                return "half_heusler"

        # ---- skutterudite: MX3 (binary, Im-3=204) or filled RM4X12 (ternary) ----
        if sgnum == 204:
            if n_el == 2 and vals[0] > 0 and abs(vals[1] / vals[0] - 3) < 0.2:
                return "skutterudite"
            if n_el == 3 and vals[0] > 0:
                if abs(vals[1] / vals[0] - 4) < 0.5 and abs(vals[2] / vals[0] - 12) < 1.0:
                    return "skutterudite"

        # ---- Bi2Te3 family: A2B3 tetradymite, R-3m (166) ----
        if sgnum == 166 and n_el == 2 and vals[0] > 0:
            if abs(vals[1] / vals[0] - 1.5) < 0.15:
                if (elements & PNICTOGENS) and (elements & CHALCOGENS):
                    return "bi2te3_family"

        return None
    except Exception:
        return None


summary_rows = []  # will collect (dataset, family, n_seebeck_datapoints) tuples


def save_family_csv(df, dataset_name, family_name):
    fname = os.path.join(OUT_DIR, f"{dataset_name}__{family_name}.csv")
    df.to_csv(fname, index=False)
    summary_rows.append((dataset_name, family_name, len(df)))
    print(f"  {dataset_name:22s} | {family_name:15s} | {len(df):6d} rows -> {fname}")
    return fname

## Section A — Ricci / Materials Project BoltzTraP (`boltztrap_mp`, matminer)

8,924 entries, self-contained (includes a pymatgen `Structure` object per row) — **no API key needed**.
Columns: `formula, m_n, m_p, mpid, pf_n, pf_p, s_n, s_p, structure`.

In [3]:
from matminer.datasets import load_dataset

df_mp = load_dataset("boltztrap_mp")
print(df_mp.shape)
df_mp.head()

Fetching boltztrap_mp.json.gz from https://ndownloader.figshare.com/files/13297511 to C:\Users\Ekta Jain\Downloads\nbenv311\Lib\site-packages\matminer\datasets\boltztrap_mp.json.gz


Fetching https://ndownloader.figshare.com/files/13297511 in MB:   0%|          | 0.0/14.020266999999999 [00:00<?, ?MB/s]

Fetching https://ndownloader.figshare.com/files/13297511 in MB: 14.020608MB [00:00, 202.42MB/s]                         

(8924, 9)


,mpid,pf_n,pf_p,s_n,s_p,formula,m_n,m_p,structure
0,mp-10070,0.865,0.0125,-421.0,760.0,BaAg(PO3)3,0.578,72.800,"[[ 0.08245398 10.58009491 11.61923254] O, [3.1..."
1,mp-10086,1.050,0.6440,-393.0,567.0,YSF,0.541,4.020,"[[2.84699546 0.94899849 0. ] F, [0.9489..."
2,mp-10096,0.977,0.4200,-305.0,471.0,Na3Sr3GaP4,0.270,1.020,"[[-2.8085287 7.06608376 1.25800196] Na, [1...."
3,mp-10103,0.889,0.0257,-416.0,871.0,YbLiAlF6,0.548,20.300,"[[2.52287112 1.45658029 7.18290524] Li, [-2.85..."
4,mp-10155,0.719,1.5700,-496.0,510.0,P2Ir,1.160,0.667,"[[0.86371064 5.14343422 4.80450637] P, [4.5512..."


In [4]:
# Classify each entry and keep only rows with a usable Seebeck value (s_n and/or s_p)
tqdm.pandas(desc="classifying boltztrap_mp")
df_mp["family"] = df_mp["structure"].progress_apply(classify_family)
df_mp["space_group_number"] = df_mp["structure"].apply(
    lambda s: SpacegroupAnalyzer(s, symprec=0.1).get_space_group_number() if s is not None else None
)

df_mp_export = df_mp[["mpid", "formula", "space_group_number", "family", "s_n", "s_p", "m_n", "m_p", "pf_n", "pf_p"]].copy()

print("Family counts (all entries, before filtering for non-null Seebeck):")
print(df_mp_export["family"].value_counts(dropna=True))

print("\nSaving per-family CSVs (rows with at least one non-null Seebeck value: s_n or s_p) ...")
for fam in FAMILIES:
    sub = df_mp_export[
        (df_mp_export["family"] == fam)
        & (df_mp_export[["s_n", "s_p"]].notna().any(axis=1))
    ]
    save_family_csv(sub, "ricci_boltztrap_mp", fam)

classifying boltztrap_mp:   0%|          | 0/8924 [00:00<?, ?it/s]

C:\Users\Ekta Jain\AppData\Local\Programs\Python\Python311\Lib\functools.py:1001: UserWarning: No Pauling electronegativity for Ar. Setting to NaN. This has no physical meaning, and is mainly done to avoid errors caused by the code expecting a float.
  val = self.func(instance)


Family counts (all entries, before filtering for non-null Seebeck):
family
half_heusler    9
skutterudite    1
Name: count, dtype: int64

Saving per-family CSVs (rows with at least one non-null Seebeck value: s_n or s_p) ...
  ricci_boltztrap_mp     | half_heusler    |      9 rows -> /content/te_family_csvs\ricci_boltztrap_mp__half_heusler.csv
  ricci_boltztrap_mp     | skutterudite    |      1 rows -> /content/te_family_csvs\ricci_boltztrap_mp__skutterudite.csv
  ricci_boltztrap_mp     | bi2te3_family   |      0 rows -> /content/te_family_csvs\ricci_boltztrap_mp__bi2te3_family.csv


## Section B — JARVIS-DFT (`dft_3d`, jarvis-tools)

~76,000 total DFT entries; Seebeck coefficients (`n-Seebeck`, `p-Seebeck`) computed via BoltzTraP for a subset (~22,190). No API key needed — downloads from NIST figshare on first call (can take a few minutes).

In [5]:
from jarvis.db.figshare import data as jarvis_data
from jarvis.core.atoms import Atoms as JarvisAtoms

dft_3d = jarvis_data(dataset="dft_3d")  # first run downloads ~1-2 GB, be patient
print("Total JARVIS-DFT 3D entries:", len(dft_3d))
print("Example keys:", sorted(dft_3d[0].keys())[:20], "...")

Obtaining 3D dataset 94k ...
Reference:https://doi.org/10.1016/j.commatsci.2025.114063
Other versions:https://doi.org/10.6084/m9.figshare.6815699


  0%|          | 0.00/48.4M [00:00<?, ?iB/s]

  0%|          | 17.4k/48.4M [00:00<07:53, 102kiB/s]

  0%|          | 69.6k/48.4M [00:00<03:43, 217kiB/s]

  0%|          | 174k/48.4M [00:00<02:02, 393kiB/s] 

...

100%|█████████▉| 48.3M/48.4M [01:38<00:00, 517kiB/s]

100%|█████████▉| 48.4M/48.4M [01:39<00:00, 477kiB/s]

100%|██████████| 48.4M/48.4M [01:39<00:00, 489kiB/s]

Loading the zipfile...


Loading completed.
Total JARVIS-DFT 3D entries: 93902
Example keys: ['Tc_supercon', 'atoms', 'avg_elec_mass', 'avg_hole_mass', 'bulk_modulus_kv', 'crys', 'density', 'dfpt_piezo_max_dielectric', 'dfpt_piezo_max_dielectric_electronic', 'dfpt_piezo_max_dielectric_ionic', 'dfpt_piezo_max_dij', 'dfpt_piezo_max_eij', 'dimensionality', 'effective_masses_300K', 'efg', 'ehull', 'elastic_tensor', 'encut', 'epsx', 'epsy'] ...


In [6]:
def jarvis_to_numeric(x):
    """JARVIS encodes missing values as the string 'na'."""
    try:
        if x is None:
            return np.nan
        if isinstance(x, str) and x.strip().lower() == "na":
            return np.nan
        return float(x)
    except Exception:
        return np.nan


# Pre-filter to entries that actually have a Seebeck value, to avoid building
# ~76,000 pymatgen structures (slow). This keeps only the ~22k relevant rows.
seebeck_entries = [
    e for e in dft_3d
    if not (pd.isna(jarvis_to_numeric(e.get("n-Seebeck"))) and pd.isna(jarvis_to_numeric(e.get("p-Seebeck"))))
]
print("Entries with at least one non-null Seebeck value:", len(seebeck_entries))

rows = []
for e in tqdm(seebeck_entries, desc="classifying JARVIS-DFT Seebeck entries"):
    try:
        jatoms = JarvisAtoms.from_dict(e["atoms"])
        structure = jatoms.pymatgen_converter()
    except Exception:
        continue
    fam = classify_family(structure)
    try:
        sgnum = SpacegroupAnalyzer(structure, symprec=0.1).get_space_group_number()
    except Exception:
        sgnum = None
    rows.append({
        "jid": e.get("jid"),
        "formula": e.get("formula"),
        "space_group_number": sgnum,
        "family": fam,
        "n_seebeck": jarvis_to_numeric(e.get("n-Seebeck")),
        "p_seebeck": jarvis_to_numeric(e.get("p-Seebeck")),
        "n_powerfact": jarvis_to_numeric(e.get("n-powerfact")),
        "p_powerfact": jarvis_to_numeric(e.get("p-powerfact")),
    })

df_jarvis = pd.DataFrame(rows)
print("\nFamily counts (JARVIS-DFT, Seebeck subset):")
print(df_jarvis["family"].value_counts(dropna=True))

for fam in FAMILIES:
    sub = df_jarvis[df_jarvis["family"] == fam]
    save_family_csv(sub, "jarvis_dft", fam)

Entries with at least one non-null Seebeck value: 23218


classifying JARVIS-DFT Seebeck entries:   0%|          | 0/23218 [00:00<?, ?it/s]

C:\Users\Ekta Jain\AppData\Local\Programs\Python\Python311\Lib\functools.py:1001: UserWarning: No Pauling electronegativity for He. Setting to NaN. This has no physical meaning, and is mainly done to avoid errors caused by the code expecting a float.
  val = self.func(instance)


C:\Users\Ekta Jain\AppData\Local\Programs\Python\Python311\Lib\functools.py:1001: UserWarning: No Pauling electronegativity for Ne. Setting to NaN. This has no physical meaning, and is mainly done to avoid errors caused by the code expecting a float.
  val = self.func(instance)



Family counts (JARVIS-DFT, Seebeck subset):
family
half_heusler     439
skutterudite      37
bi2te3_family      4
Name: count, dtype: int64
  jarvis_dft             | half_heusler    |    439 rows -> /content/te_family_csvs\jarvis_dft__half_heusler.csv
  jarvis_dft             | skutterudite    |     37 rows -> /content/te_family_csvs\jarvis_dft__skutterudite.csv
  jarvis_dft             | bi2te3_family   |      4 rows -> /content/te_family_csvs\jarvis_dft__bi2te3_family.csv


## Section C (optional) — Full Ricci tabular (47,737 entries) via live Materials Project lookup

This is the *complete* Ricci et al. dataset (bigger than `boltztrap_mp` in Section A, but it doesn't ship with structures — only `mpid`s). To classify by family we need each structure's space group, which we fetch live from the Materials Project API in batches.

**Requires a free API key:** create one at https://materialsproject.org/api and paste it below. Skip this section if you don't want to sign up — Sections A and B already give you real counts.

In [7]:
MP_API_KEY = ""  # <-- paste your Materials Project API key here, or leave blank to skip this section

if MP_API_KEY:
    from matminer.datasets import load_dataset
    from mp_api.client import MPRester

    df_ricci_full = load_dataset("ricci_boltztrap_mp_tabular")
    print(df_ricci_full.shape)
    print(df_ricci_full.columns.tolist())

    # mpid column name may be 'mpid' or 'task' depending on matminer version -- check both
    id_col = "mpid" if "mpid" in df_ricci_full.columns else "task"
    all_ids = df_ricci_full[id_col].dropna().unique().tolist()
    print(f"Unique material IDs to look up: {len(all_ids)}")

    symmetry_records = []
    batch_size = 1000
    with MPRester(MP_API_KEY) as mpr:
        for i in tqdm(range(0, len(all_ids), batch_size), desc="querying MP summary"):
            batch = all_ids[i : i + batch_size]
            docs = mpr.materials.summary.search(
                material_ids=batch,
                fields=["material_id", "formula_pretty", "symmetry", "structure"],
            )
            for d in docs:
                symmetry_records.append({
                    "mpid": str(d.material_id),
                    "formula_pretty": d.formula_pretty,
                    "space_group_number": d.symmetry.number if d.symmetry else None,
                    "structure": d.structure,
                })

    df_sym = pd.DataFrame(symmetry_records)
    print("Fetched symmetry/structure for", len(df_sym), "materials")

    df_sym["family"] = df_sym["structure"].progress_apply(classify_family) if hasattr(df_sym["structure"], "progress_apply") else df_sym["structure"].apply(classify_family)

    merged = df_ricci_full.merge(df_sym[["mpid", "space_group_number", "family"]], left_on=id_col, right_on="mpid", how="left")

    seebeck_cols = [c for c in merged.columns if "S." in c or "seebeck" in c.lower()]
    print("Detected Seebeck-related columns:", seebeck_cols)

    for fam in FAMILIES:
        sub = merged[
            (merged["family"] == fam) & (merged[seebeck_cols].notna().any(axis=1))
        ]
        save_family_csv(sub, "ricci_full_mp_api", fam)
else:
    print("MP_API_KEY not set -- skipping Section C. Get a free key at https://materialsproject.org/api")

MP_API_KEY not set -- skipping Section C. Get a free key at https://materialsproject.org/api


## Summary table (dataset × family → number of Seebeck data points)

In [8]:
summary_df = pd.DataFrame(summary_rows, columns=["dataset", "family", "n_seebeck_datapoints"])
pivot = summary_df.pivot_table(index="family", columns="dataset", values="n_seebeck_datapoints", fill_value=0, aggfunc="sum")
pivot = pivot.reindex(FAMILIES)

summary_path = os.path.join(OUT_DIR, "summary_table.csv")
pivot.to_csv(summary_path)

print("Saved summary table to:", summary_path)
pivot

Saved summary table to: /content/te_family_csvs\summary_table.csv


dataset,jarvis_dft,ricci_boltztrap_mp
family,,
half_heusler,439,9
skutterudite,37,1
bi2te3_family,4,0


In [9]:
# Zip everything up for easy download from Colab
import shutil

zip_path = shutil.make_archive("/content/te_family_csvs", "zip", OUT_DIR)
print("Zipped output ready at:", zip_path)

try:
    from google.colab import files
    files.download(zip_path)
except Exception as e:
    print("Not running in Colab (or download blocked) -- files are at:", OUT_DIR, "and", zip_path)

Zipped output ready at: C:\content\te_family_csvs.zip
Not running in Colab (or download blocked) -- files are at: /content/te_family_csvs and C:\content\te_family_csvs.zip
